# Does the detector have a blind spot for unseen defect types?

`defect-inspect` pairs a supervised YOLOv8n with PatchCore, an unsupervised backstop
gated on low detector confidence. The stated rationale: a supervised detector can only
report classes it was trained on, so an *unseen* defect type produces no detection and
the part passes.

We measured that on low-confidence frames, simply lowering the detector threshold
recovers 96% of them — but every test defect was one of the six trained classes, so that
measurement **cannot see** the blind spot. This notebook runs the experiment that can.

**Protocol** (after DRA, Ding/Pang/Shen, CVPR 2022 — 'hard setting'): hold out one class
entirely, train on the rest, test only on the held-out class. Six folds, one per class.

**Two questions, and they are different:**
1. *Operational.* On an image containing only an unseen defect, does the detector fire at
   all? For a PASS/FAIL ejector a **wrong label is still a correct ejection** — only
   silence is a failure. This is the number that decides whether the blind spot is real
   for this system.
2. *Diagnostic.* Which score best separates seen from unseen — max softmax (what the gate
   uses today), max-logit, or energy? IJCV 2024 reports max-logit and energy beat MSP
   across OOD benchmarks, and switching costs nothing.

Runtime: six ~10-minute training runs on a T4, about 75 minutes total.
**Session options → Accelerator → GPU T4 x2.**


In [ ]:
import os
os.makedirs('/content', exist_ok=True)
os.chdir('/content')
import torch
print('torch', torch.__version__)
assert torch.cuda.is_available(), 'No GPU. Session options > Accelerator > GPU T4 x2.'
print(torch.cuda.get_device_name(0))


In [ ]:
%pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)


## Dataset

Same NEU-DET fetch and the same pinned split manifest as the resolution experiment, so
the folds below are carved out of exactly the split the deployed model was trained on.


In [ ]:
import subprocess, pathlib
UP='https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection.git'
raw=pathlib.Path('/content/NEU-DET-raw')
if not raw.exists():
    subprocess.run(['git','clone','--filter=blob:none','--sparse','--depth','1',UP,str(raw)],check=True)
    subprocess.run(['git','-C',str(raw),'sparse-checkout','set','IMAGES','ANNOTATIONS'],check=True)
imgs=sorted(raw.rglob('*.jpg')); anns=sorted(raw.rglob('*.xml'))
print(len(imgs),'images',len(anns),'annotations')
assert anns, 'No XML -> this is NEU-CLS, the wrong dataset.'


In [ ]:
import base64, gzip, json
MANIFEST_B64='H4sIAOdDrWoC/22d3a7duA2F3yXXU8Be/u+rFIMgyKRtgCBTJJletOi798z2QY6+tb+7TdkWSZlckiWS+7/vfnz6/uPdX//27uO3D//5/PUf7+dpfvfLG0ViAhVQy0gt10it6GXdR2pDn/s2Ugf6PHDtGq9lmkDNoAJqBTXKmRnXco4U2G14bAfz6wA13jl2v4+jMHI6RzWvP7v+/PXjlz++f/796/s5c9Fb0QfppcjqbtlJbyvpI0VXf2exOy/Q94sY6CLD7rI0fZJeKW7OFF3sLz6/kv3GpzfqsnModl49SJYgJ99KiXX92fO/Pvz4+M9P319ewQbqGKkddx4TqPFapgPUNVLzBArPBcQOCl0+nOEnteC5FXeuuHMb1VvG37gwcl7GDkZp1/GubR6J8co+EsfI8xh7Psfbrkdvn3/8+PTb++9/fPv7h4+fXpxpscbVGk9pfDjbU6M9rm27NG4m0T3E1XhYmwl0mugP1KrGTNYmUsaYZ7YuV2uLNYriN+5242F8Tmu8RPZlkj4Xu1F0XOVZe2PS3WNu+/b7ly+ffvvL56/vv3/88OXlNTxmrqfWx5z81Kq3Pgb4qXXRexcV4YF7z62ntT587an1nLRVZThNhkzGLarxDW9PrXqrKnzPOU+tm/agwxB9l7ddPrVe1u9thN26mmar8Vrtne1mNYfd+UDJ7y8rkNeZ5jF6I72TXoqsxx9eMdJz0dX9Wv09tBnoB9yN9EWat6fphexS7PIY5oHeKG72en6v54+6ftT1k+Lnqv6vuv8q+S+qu/DxjYO981W9IN6vv7z78e3D569Y8o/raizHJ1JY8E9YnGNZPU9Yqk9Y8E8HqBMfEfzcwHMz7oQogZjhNYgZiBmIGbALxAyZ4+uGHz4QZYE+CwRb+BwEWyDYAlFw4wp2KxisvBNdrtBnhT64cSPFTzc8hks7v+owDDsU3yHlDlV3SHmgzwN9HtD8QJ8HJDvA4SAHKIRLJ5ifYH6C+QnmJ3q50MuFXi70cqGXi5++UAjfm/NFfvhOxYcvCH5Lb6DwOQt3DT6zgRyZeQ284TGBvQVmdIP3G0Xm0A0MqB3FrDvBHAgQIEDg2IFjB44dOHbgylnAYSEH7jdAPWxMwM2z8hrYrWAANw82Xu5p7o3ibgc4ABGygQO0g9cHXh94fXZ2gnHAJUBA4PTBZlEAAcHWUQAIOcgBzEmAOZw+J5gDAoINnZxgfoL5SX54XSS444SBBo6E22RAlQArxvsWzPuYmmDOsFgYLOYlbP/BeNeRKwwZlovZClY8SgODhj1vI0/YNkwblo2pDlaOaQ8THSx+H5nC+mH82PGEI8APxgvwAsyD8Ag4BPwB7gD7h/ljOoT1Y/6DJ8AR4AcwfNg9DB2/Rzawa0yImA8xHcLCL27D1U5q7YxOfbl2QqfaCZ1q53Sqjdmpdlqn2jmdzqJL2BKnxX26XuL2RvBc4szFfi721V16W7q6T41Gil2K3VLiL8Vv6X3s6n/pbe/uv9Spx9dSpzaK73X1SJc4a/dXL7/2re9P2pEu8erxrcTbSrzayL53CUe6d/mL/VajtZU4e/Hfi99e/GrrfN6Lf+2l38v88ZSh9Dv6VKLG56j+a3P+XuSPdOlXl89Stzb357MPQUq82v2/PwEGurj3Ec9V2l/F/ir2V7G/Svur2dchSZ3YFNl0ndAU7KVgLwV7KZhLwVwK11LAk8KxlCencO3+gBjp4lfdt/pP14td4VoK157Ot6r7grkUrN3fDiNd7Ar2UjDX0hbMpWAuBWtZ+/k+jit2JW3BVAqWUjCUgqEUDPVYFyrdnxsjXYNbqJS9+ytxCpWylzh1+1HiFIiljlJTR6kpkEuBXI7mV+IVqLUlFqjlLPEK1FKgdn+2jHSJUyCXPokt9gVqKVBLgVoK1FIgxqfLzZYCsVpslBPWUqNcsjywHLCWHeWO7KmcsZYY5Yq1oKj1Q/kh763FQzllHfhv9Sy1LYodl7vt7KmcrXyrXKuO2smn/KrcqryoVgblI+UDZdJl0XWGT6Fqzi7rLuO+ODQXZbzGs/vxjPvejH+jeC2gEDcw7aAQYTCdoMAbDBBEMNc1MJ8XUBBlRrDDDMFmCDZDsBmCgV0gWCBYeCcECwQLBEMgBPVZwG4BO0RF3N8sbxSDPsgA2uHGFexWsFvBbgW7lb1AO4RRzAjXuL9EflJ4bAO7Dew2sNv4HNhteMsbmG9gjsd2jMOOcdgh2A7mO3thdA2Y72B+oE8EkdyfH28UOBzggHgTWvQJhU4odILdibE9wRwxLHc01hsFftDuArsLul5gfoHdBXYX2AG34JQBVAVQdX9QvFEMatpAIToJXc6k0CWMODM7QeASXlAwYqFC5F5BVOAOXEnQJ24ErgS4EmBHgB1BwFgWRmmBHaAkgJKsjOeCZACPADwCuKCuG9ghWiuAkgA8ArjIxmAysMONAIgAEu6jhzcK7AAX2dknxhYAweC4A9oBOwJ8CPAhcPv7ROGNgtAAgQAEAhAI3P418Oln7AAIMAcIBCAQuH3g9vcZwr8QnPAzjg8LFpg0LBomDJuFkcJGMb/BQjGfwT7Hu2CBmMtgjZjJMHXBMDEfwfZgbLA1GBd+j6MGy4JhIf4TUxIs7hhHAOGPIxuYHmwNpoYJBoYGO7vGrmFYCH3F3AIbs/BDC9C0YL3XhXE3xhotmnKyYMzJQiwni7GcLJxyMn1MTG802TVmdDYxNRR0NjEtRFOZWyjo69K7G/VxC221d7FYl4t1uWiXprkFab4urLvRGFkg6utquxvtZawqkr2M1V6GdbmZQpsptGlIsUm0mUSbSWRdWlzufUjw1GgDt5uWuzE/7A0d1udhch4aJG2qW1Dx6xK+G23gT9P9NDlP0/004U8T3uKc1asvk12D4C8b48sUukyhyxSyyOv7nKJjtC3kXOPQZ2vUxy3m3PA9BuUxKI/BdgzRYrAdc+vMGllvIpmWGqxvCB1/3GSPiWlY/po60o02csbH8iZiU0FsKohlWNwnKU+NJrtNGq9fUp2TYHLaTJJV8xxMJMv6iM0P+jJsKohNBbG0hNcPsW5URjZIm4ppI2dK7prhYRLtxnw35jZpqG0emkliA2dTzn2W89Rosh82moeKZAppm8lu09DrB2Y3muw2N8VmnFj6kHq6zTixGSc248RmnNjk8vp92o0qpsi5WE6QtNl9MmiGO4vIYgAj3Rm62DLVVqmGN7YcNbSxxaiBjWHNJvIZ0himGKQYouzCdxe+BjK23DSIMYSR2wxKDDQMHmyVajhgy1G5zZDBMMAgwBDAFqe2NjWgsJWpNYl45vq21jQ0sJWmAYThw2UZW5p7pxluk9+qGXmTp//t2qoZeZojd288PCcQqryeQThrTp9m2d3bD8+tqsWsWsyqxaxaxNMgPQ9SJYtKFpUsKllUMh2yRcVd/F5P21RxFxV3UcFWffGrSrbqQK4qr2Yh3ueMz60q76rDu6oWymxT1TRt8z6tfG5VcTcVd1NxtYNdBdNszPsk87lVx3z3dF2VQXU4VLBDBTtUsEPH0VOZD5fB84j1teutp4p76iicKsLp/aoIlw7ZpTJoZu+9R/LcqkN2qRaXanGpFjqDaSbzpI3eOmurd7toq+ZoT577vWvroa2qroqrU2Bmv1eVUBiKTlXx4XXJPFc9KkM82V0l08kuOtlFJ7voZOeCLaqwToFZvAcVVye76GTnY6PzV1a/V0VYvT6ANuoobCrC5j2oCDonReek7F64QGXQKSU6pUTrQrg16UQTnWiiE020sMR9sv7cqsOgRSjuvY/nVrUGL02hM01Or/WgjSrYpYLphBCdEKITwqLFJqxRy1Lo4zZYWllDfVerbag/a/ULrZNhYqovqyvrmtP0Ue/WBad6vC5C1bPVsTcT05TU9aeigrq/+rnWPdlNIq0vYhIpHCga6KpTF50KELq4NH20Qo0ihgKG4oXCha5LdVmquKIIomtSXZIq1OgyVVepikkKSYpIVc2kSsNMVTpm6utVOmaq0jFTVaqZqpTMVKVqppKm2M0lztP1Eqcr48xVuWYu8apUzb0bMtIlXkqelDwp/lmLLnlS/NOFgIr/UvyrtM69uTHSxX/pQkLFfyl+9fjadKmzVvdrqdN1hEqbrbTZSput2G/Fbuu6RcW+bq+6QnPVFbpDJUa6xN+7v2K/N/sum1TGU3WM5qpjdH/8j3TJc5Q8R/E7a7jP4n8W/7P4Vx2l++t+pEvfs/g3WeJcxb7KMt0f8iNNdasI1FRk01UDaqoaUYVjKRxL4VgKxwoXUjiWua+X9IVjKUdMeVLKtFM4lhr7qi+WHq2n6yVOwVoK1lL1ypKuCFb8qqRWCsZSMHbHFox08VuaXw1HwVyqYFrWrlhW/KpA2h2UPVYwK7K6L5xL4Vy2LohW6hXOpXAuVZ/tjtQe6ZKvgCwFZCkgSwFZv93CqRROpXAqhVM1B6ZgKwVbKdjK2eXfSpyCrRRMpcnqvmDoPhEf6dLuojoUrhyh7L7MvKy8jLxsvEycQpe9V33Asv4y/rL9Mn3eW5X8yu7L7KvKYBl92XiZOO8te98pRU37NcuXb5Tpl+UfZFQTeM3f5QblBVUCsnyCF8sDygHK/sv8axIvZ6gpnELUhF1lFGu6LjcpL6kSiy/fJL/+8u7fH76wLDpqw7EKH0vmoWjZjDJlc1gQEIXcFtbkQ5U8lEOaUc9oRg2jeeOd6BMlie4V4FvtO1StA3MUsJpRx2c++Ry6RMmqibXVUJaK5c1Y6mpheTM8h0ryQZUoFnbDgAWlnYIhCoYoKM6UneXNICbKMAWVk3KyPhf6RO0i1JhDiTlUmEOBOdgQCsqhFByGCiqPF6ATyqShNBmKinX9kalLE9X1rm5TJTrmdK2hqpeydHGdrl5T17v4TBePKbKLv1Q1lqqDMF9d4KTrqXQBkq6IX2UeqhhNnkrm1/1dhqILLyxd46Pkq8oCWbsoRxVuqFpDqQIC2boISFfZ6LIWXVeirldmfa6+XgUZqlBC1VzgzWU5VZ+kypFUNZKqNlIlkKq4R9XyqNIdRXHAqy5HleUoa6wqGReT+0Egy3di9jvyepE0OC/MFmcyN3O0mTMNAunHyFCbkYk2n0xbBgP+jQLzEJE3zFxkJvkydZd5tszPZRYsCOav4rGNKaRIuUT+eZD2F2ScB+l9OfgckjqR1ZeTqZq4huELcvGQi4yiBhgTJBQjZxh2AbNABi/+tGG8CyODvFvkzyJ9FvmyyEbEhfH5S3P1rM3yVTRjTfOhLOR8tiDpedEsOEuhWfQfJzRnzBJ4Nk28svwfC6N9LavQjabmrulUxujQJCnNPrIIb0vmzKT/RGFtFp0+a2qMhcZrYP6iyS1256rpFMZ909QJ08jeWyzWOfYyY6HIsQDgWARwLD7XUngsg8eyZSwvRoSxZBUbMstQtGGwBBIzXrNdy+qwtAzLoLBkCQ8zmzz22KN2NYRVO4jHpXr8qIdTKjc9Cp83D+nUGLxdue0eMuh/SOIxeNrooV8acaEBiln8r0c84EhjQTQ8NxqeGw3EjYbcRgMdoq8tGqwQfUEeZ6ZRCNEo2Ogbjr7haDBANNjUg9o0gDR69p/Tu9V+1Vuj5+rRg/Vo8KaG0GmsnJqNWo2+cnVJDUfTWDJ9hxpJprFd+lbqxK5jA+pAburD9j78rrPmOn5c+jC5uNdm8ly7vPPex6l1/ajjyv7TnSZL+rNOV2v7c64tzbn2/eeKu8jUB5Z1bDH1GVsde/QxSZ9b9J/sFFm91cl91j5S6iOjIqu7Gvscfb2PjPrMp+jaeq6whzoerP8XqqCF0rQUraO3OmmrCIE6JyuKKtehWA1AnXHVkVQdQb1Y1q//+z+xZGtyR3QAAA=='
SPLIT=json.loads(gzip.decompress(base64.b64decode(MANIFEST_B64)))
assert (len(SPLIT['train']),len(SPLIT['val']),len(SPLIT['test']))==(1236,353,181)
print({k:len(v) for k,v in SPLIT.items()})


In [ ]:
import xml.etree.ElementTree as ET, shutil
from pathlib import Path
CLASSES=['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
IDX={c:i for i,c in enumerate(CLASSES)}
def to_yolo(b,w,h):
    xmin,ymin,xmax,ymax=b
    xmin,xmax=sorted((xmin,xmax)); ymin,ymax=sorted((ymin,ymax))
    return ((xmin+xmax)/2/w,(ymin+ymax)/2/h,(xmax-xmin)/w,(ymax-ymin)/h)
assert to_yolo((0,0,100,200),200,200)==(0.25,0.5,0.5,1.0)

img_by={p.stem:p for p in imgs}; ann_by={p.stem:p for p in anns}
LBL={}   # stem -> list of 'cls xc yc w h'
CLSOF={} # stem -> frozenset of class names present
for split,stems in SPLIT.items():
    for stem in stems:
        r=ET.parse(ann_by[stem]).getroot(); sz=r.find('size')
        W,H=int(sz.find('width').text),int(sz.find('height').text)
        lines=[]; names=set()
        for o in r.findall('object'):
            nm=o.find('name').text.strip(); names.add(nm); b=o.find('bndbox')
            xc,yc,bw,bh=to_yolo(tuple(float(b.find(k).text) for k in ('xmin','ymin','xmax','ymax')),W,H)
            lines.append(f'{IDX[nm]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
        LBL[stem]=lines; CLSOF[stem]=frozenset(names)

tot=sum(len(v) for v in LBL.values())
print('boxes total:',tot)
assert tot==2887+801+438, tot
multi=sum(1 for s in CLSOF if len(CLSOF[s])>1)
print(f'{multi} images carry more than one class -- this is why the hold-out is by LABEL, not filename')


### Why the hold-out is by label content, not by filename

NEU-DET filenames name one class, but **121 of 1770 images contain boxes of a second or
third class** (49 are `inclusion + scratches`). Holding out `scratches` by filename would
leave 49 scratches boxes in the training set through images named `inclusion_*`, and the
class would not be unseen at all. Every fold below excludes any image containing *any* box
of the held-out class.

The OOD test set is stricter still: only images whose label set is *exactly* the held-out
class. An image with both a seen and an unseen defect cannot answer the question, because
the detector may fire on the seen one.


In [ ]:
def build_fold(held):
    """Write a YOLO dataset excluding every image containing `held`. Returns paths+counts."""
    root=Path(f'/content/fold_{held}')
    if root.exists(): shutil.rmtree(root)
    keep={}
    for split in ('train','val'):
        stems=[s for s in SPLIT[split] if held not in CLSOF[s]]
        keep[split]=stems
        (root/split/'images').mkdir(parents=True,exist_ok=True)
        (root/split/'labels').mkdir(parents=True,exist_ok=True)
        for s in stems:
            shutil.copy(img_by[s], root/split/'images'/f'{s}.jpg')
            (root/split/'labels'/f'{s}.txt').write_text('\n'.join(LBL[s])+'\n')
    # the model still has 6 output slots; the held-out one simply never appears in training
    (root/'data.yaml').write_text(
        'train: train/images\nval: val/images\nnames:\n' +
        ''.join(f'  {i}: {c}\n' for i,c in enumerate(CLASSES)))
    ood=[s for s in SPLIT['test'] if CLSOF[s]==frozenset({held})]
    idd=[s for s in SPLIT['test'] if held not in CLSOF[s]]
    # no leakage, by construction -- assert it anyway
    for s in keep['train']+keep['val']: assert held not in CLSOF[s]
    return root, keep, ood, idd

for c in CLASSES:
    r,k,ood,idd=build_fold(c)
    print(f'{c:<17} train {len(k["train"]):>5}  val {len(k["val"]):>4}  pure-OOD test {len(ood):>3}  ID test {len(idd):>4}')


## Train six models

Identical settings to the deployed model apart from the missing class: YOLOv8n from COCO
weights, imgsz 320, 100 epochs, patience 25, seed 42, `deterministic=True`.


In [ ]:
from ultralytics import YOLO
import time
MODELS={}
for c in CLASSES:
    root,_,_,_=build_fold(c)
    t0=time.time()
    m=YOLO('yolov8n.pt')
    m.train(data=str(root/'data.yaml'), imgsz=320, epochs=100, patience=25, seed=42,
            deterministic=True, device=0, project='/content/runs', name=f'no_{c}',
            exist_ok=True, verbose=False)
    MODELS[c]=f'/content/runs/no_{c}/weights/best.pt'
    print(f'{c}: {(time.time()-t0)/60:.1f} min -> {MODELS[c]}')


## Score every test image

Three image-level scores from the detector's own head, no second model:

- **MSP** — max class probability. What the PatchCore gate uses today.
- **max-logit** — `log(p/(1-p))` of that same max probability. YOLOv8's head applies a
  sigmoid per class, so the logit is recoverable exactly.
- **energy** — `-logsumexp` over the six class logits of the highest-scoring box.

A *low* score should mean 'unfamiliar'. AUROC below is computed so that higher = better
at separating ID from pure-OOD.


In [ ]:
import numpy as np
from ultralytics import YOLO

def logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    return np.log(p / (1 - p))

def score_image(model, path):
    """Three image-level ID-ness scores. HIGHER = more familiar, for all three,
    so they can go into the same AUROC call without sign confusion.

    energy is conventionally E(x) = -logsumexp(logits), where ID samples score
    LOWER. We return -E = logsumexp(logits) so its direction matches MSP and
    max-logit. Getting this backwards silently reports separation in reverse.
    """
    r = model.predict(path, imgsz=320, conf=0.01, verbose=False)[0]
    if not len(r.boxes):
        # Silence is the maximally unfamiliar outcome; float('-inf') would break
        # the rank statistics, so use the floor the clip already implies.
        return 0.0, float(logit(0.0)), float(np.log(np.exp(logit(np.zeros(len(CLASSES)))).sum())), 0
    conf = r.boxes.conf.cpu().numpy()
    cls  = r.boxes.cls.cpu().numpy().astype(int)
    msp  = float(conf.max())
    # YOLOv8's head is sigmoid-per-class; rebuild a per-class vector from the
    # boxes, taking the strongest evidence seen for each class in the image.
    vec = np.zeros(len(CLASSES))
    for c_, p_ in zip(cls, conf):
        vec[c_] = max(vec[c_], p_)
    lg = logit(vec)
    return msp, float(logit(msp)), float(np.log(np.exp(lg).sum())), len(conf)

def _rankdata(a):
    a = np.asarray(a, float)
    order = a.argsort()
    ranks = np.empty(len(a), float)
    ranks[order] = np.arange(1, len(a) + 1)
    for v in np.unique(a):            # average ties
        m = a == v
        if m.sum() > 1:
            ranks[m] = ranks[m].mean()
    return ranks

def auroc(pos, neg):
    """P(score(ID) > score(OOD)). 0.5 = no separation, <0.5 = inverted."""
    pos = np.asarray(pos, float); neg = np.asarray(neg, float)
    if not len(pos) or not len(neg):
        return float('nan')
    ranks = _rankdata(np.concatenate([pos, neg]))
    rp = ranks[:len(pos)].sum()
    return float((rp - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))

# self-checks: a broken AUROC would report plausible nonsense
assert auroc([5,6,7],[1,2,3]) == 1.0
assert auroc([1,2,3],[5,6,7]) == 0.0
assert auroc([1,2,3],[1,2,3]) == 0.5
assert abs(logit(0.5)) < 1e-12
_hi = np.log(np.exp(logit([0.9,0.1,0,0,0,0])).sum())
_lo = np.log(np.exp(logit([0.2,0.2,0.2,0.1,0.1,0.1])).sum())
assert _hi > _lo, 'energy score must be higher for the more confident vector'
print('scoring and AUROC self-checks passed')


In [ ]:
THRS=(0.35,0.25,0.10,0.05)
rows=[]; auc_rows=[]
for c in CLASSES:
    _,_,ood,idd=build_fold(c)
    m=YOLO(MODELS[c])
    S={'ood':[], 'id':[]}
    for tag,stems in (('ood',ood),('id',idd)):
        for s in stems:
            S[tag].append(score_image(m, str(img_by[s])))
    # operational: does it fire at all on a purely-unseen defect?
    for thr in THRS:
        fire_ood=np.mean([1.0 if msp>=thr else 0.0 for msp,_,_,_ in S['ood']])
        fire_id =np.mean([1.0 if msp>=thr else 0.0 for msp,_,_,_ in S['id']])
        rows.append({'held_out':c,'thr':thr,'fire_on_unseen':round(fire_ood,3),
                     'fire_on_seen':round(fire_id,3),'n_ood':len(ood),'n_id':len(idd)})
    auc_rows.append({'held_out':c,
        'MSP':round(auroc([x[0] for x in S['id']],[x[0] for x in S['ood']]),3),
        'max_logit':round(auroc([x[1] for x in S['id']],[x[1] for x in S['ood']]),3),
        'energy':round(auroc([x[2] for x in S['id']],[x[2] for x in S['ood']]),3)})
    print('done', c)

import pandas as pd
df=pd.DataFrame(rows); auc=pd.DataFrame(auc_rows)
df.to_csv('/kaggle/working/holdout_firing.csv',index=False)
auc.to_csv('/kaggle/working/holdout_auroc.csv',index=False)
print()
print('FIRING RATE on images containing ONLY an unseen defect')
print(df.pivot(index='held_out',columns='thr',values='fire_on_unseen').to_string())
print()
print('for reference, firing rate on seen-class images')
print(df.pivot(index='held_out',columns='thr',values='fire_on_seen').to_string())
print()
print('AUROC separating seen from unseen (0.5 = no separation)')
print(auc.to_string(index=False))


### How to read it

**Question 1 — is the blind spot real?** Look at the firing rate on unseen defects at
0.10. If it is high, the detector still ejects parts carrying defect types it was never
trained on, and the backstop is not needed for that job either. If it is low, the blind
spot is real and PatchCore earns its 110 ms.

Note the asymmetry in what counts as success: a **wrong label on an unseen defect is
still a correct ejection.** Only silence fails.

**Question 2 — which gate score?** Whichever of MSP / max-logit / energy has the highest
AUROC should be what gates PatchCore. If max-logit or energy beats MSP, that is a free
improvement to `anomaly.gate_conf_below`'s underlying signal.

**Limits, to state in the paper.** 24–30 pure-OOD images per fold is small; report the
count beside every number. Each fold trains on ~1000 rather than 1236 images, so these
models are not directly comparable to the deployed one — compare folds to each other.


In [ ]:
import shutil, os
for c in CLASSES:
    d=f'/content/runs/no_{c}'
    if os.path.exists(d):
        shutil.copytree(d, f'/kaggle/working/no_{c}_run', dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('*.pt'))
    if os.path.exists(MODELS[c]): shutil.copy(MODELS[c], f'/kaggle/working/best_no_{c}.pt')
print(sorted(os.listdir('/kaggle/working')))
